In [7]:
import sys
import numpy as np

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

sys.path.append('../../')
from Rain import Rain
sys.path.pop()

'../../'

In [8]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.legacy.Adam(learning_rate=0.001),
}

In [9]:
def get_train_data():
    return np.load('../../data/MNIST/train_data.npy'), np.load('../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../data/MNIST/test_data.npy'), np.load('../../data/MNIST/test_labels.npy')

def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)
    
    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size:])
            y_train_partitions.append(y_train[i * partition_size:])
        else:
            X_train_partitions.append(X_train[i * partition_size:(i + 1) * partition_size])
            y_train_partitions.append(y_train[i * partition_size:(i + 1) * partition_size])

    return X_train_partitions, y_train_partitions

In [10]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation('softmax'))
    return model

In [11]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config['partitions'])

In [12]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Provisioner created successfully


In [13]:
rain.setup_vms()

Created resource group: Rain_resourcegroup
Created virtual network: Rain_vnet
Created network security group: Rain_nic-nsg
Network setup completed
Created public IP address: Rain_nic-Rain-vm1-ip
Created network interface: Rain_nic-Rain-vm1
Created virtual machine: Rain-vm1
Created public IP address: Rain_nic-Rain-vm2-ip
Created network interface: Rain_nic-Rain-vm2
Created virtual machine: Rain-vm2


('20.231.55.180', '20.232.17.138')

In [16]:
rain.delete_vms()


Deleting virtual machine: Rain-vm1
Deleting network interface: Rain_nic-Rain-vm1
Deleting public IP address: Rain_nic-Rain-vm1-ip
Deleting virtual machine: Rain-vm2


In [ ]:
model = rain.train_centralized_sync()

In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 3ms/step - loss: 0.0745 - accuracy: 0.9808

Test accuracy: 98.1%
